# Train and Tune Random Forest Model

In [ ]:
from pathlib import Path

import optuna.visualization as vis
import pandas as pd

from config.config import Config
from src.data import time_series_split
from src.models.factory import Experiment
from src.models.random_forest import RandomForest
from src.plots import plot_forecast_diagnostics, plot_forecast_overlay, plot_test_overlay, plot_val_overlay, plot_val_test_overlay
from src.runners import run_experiments
from src.utils import set_seed

In [ ]:
cfg = Config(Path("../config/config.yaml"))
SEED = cfg.runtime.seed
HORIZON = cfg.runtime.horizon
rng = set_seed(SEED)

In [ ]:
df_full = pd.read_csv(Path(cfg.data.processed_dir) / "features_full.csv")

In [ ]:
MODEL_NAME = "random_forest"

experiments = [
    Experiment(
        name=MODEL_NAME,
        build=lambda horizon, seed: RandomForest(horizon=horizon, random_state=seed),
        include_sentiment=True
    )
]

In [ ]:
results = run_experiments(df_full, Path(cfg.data.processed_dir), experiments, HORIZON, SEED, n_trials=100, n_splits=5)

In [ ]:
train, val, test, forecast = time_series_split(df_full, train_ratio=0.7, val_ratio=0.15, horizon=HORIZON)

In [ ]:
plot_test_overlay(test, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_val_overlay(val, results, Path(cfg.data.fig_dir) / f"val_{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_val_test_overlay(val, test, results, Path(cfg.data.fig_dir) / f"val_test_{MODEL_NAME}_actual_vs_predicted_adj_close.png")
plot_forecast_overlay(test, forecast, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_forecast.png")
plot_forecast_diagnostics(forecast, test, results, Path(cfg.data.fig_dir) / f"{MODEL_NAME}_forecast_diagnostics.png")

In [ ]:
pd.DataFrame(results[0]["best_params"], index=[0])

In [ ]:
pd.DataFrame(results[0]["metrics"]["test"], index=[0])

In [ ]:
study = results[0]["study"]

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()
vis.plot_slice(study).show()
vis.plot_parallel_coordinate(study).show()
vis.plot_contour(study).show()
vis.plot_edf(study).show()

In [ ]:
from src.train import ModelTrainer
import pandas as pd
from pathlib import Path

model_path = Path(cfg.data.models_dir) / f"{MODEL_NAME}.pkl"
model, preprocessor, _, _ = ModelTrainer.load(str(model_path))

In [ ]:
x_train_path = Path(cfg.data.processed_dir) / f"X_train_{MODEL_NAME}.parquet"
X_train = pd.read_parquet(x_train_path)

In [ ]:
feat_names = preprocessor.get_feature_names_out(X_train.columns)
importances = pd.Series(model.model.feature_importances_, index=feat_names).sort_values(ascending=False)

In [ ]:
def base_feature(n):
    tail = n.split("__")[-1]
    return tail.split("_", 1)[0]

grouped = importances.groupby(importances.index.map(base_feature)).sum().sort_values(ascending=False)
print(grouped.head(20))